# 09 · Explain a CNN prediction

**Level:** advanced. **Prerequisites:** tensors, CNNs, a training loop, and validation.
**Live demonstration:** approximately 30–45 minutes of teaching, plus training/download time.

By the end you can compute input saliency, implement Grad-CAM, test an explanation with
occlusion, and explain why a colorful map is not proof that a model is correct.
This notebook trains its own small Fashion-MNIST model; no earlier checkpoint is needed.
The default uses 4,000 training examples. Set `QUICK=False` for a larger experiment.
Accuracy depends on training and is intentionally not pre-filled.


## 1. Prepare the environment

Run the setup notebook first to install dependencies. Then use **Restart Kernel and Run All**.
The next cell finds the project folder and selects CUDA when it is available.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


In [ ]:
from torch import nn
import torch.nn.functional as F
from torchvision import transforms

cu.seed_everything(42)
N_TRAIN = 128 if SMOKE else (4000 if QUICK else 20000)
N_VAL = 48 if SMOKE else (800 if QUICK else 3000)
N_TEST = 48 if SMOKE else (1000 if QUICK else 3000)
EPOCHS = 1 if SMOKE else (3 if QUICK else 8)


## 2. Keep input coordinates easy to interpret

`ToTensor` maps grayscale pixels to `[0, 1]`, so zero really means a black pixel in this
lesson. Normalized inputs would require undoing normalization for display and choosing
an appropriate occlusion value. The training and evaluation transforms are identical here.


In [ ]:
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_transform=transforms.ToTensor(), eval_transform=transforms.ToTensor(),
    train_limit=N_TRAIN, val_limit=N_VAL, test_limit=N_TEST, seed=42,
)
train_loader = cu.loader(train_ds, batch_size=64, shuffle=True)
val_loader = cu.loader(val_ds)
test_loader = cu.loader(test_ds)
print('Training / validation / test:', len(train_ds), len(val_ds), len(test_ds))
print(class_names)


## 3. Train a model to investigate

The helper uses cross-entropy on logits and restores the checkpoint with the lowest
validation loss. We will explain its decisions, including any mistakes.

**Ask the audience:** does a correct prediction imply that the model used the right evidence?


In [ ]:
model = cu.SmallCNN(num_classes=len(class_names), in_channels=1).to(DEVICE)
history = cu.fit(model, train_loader, val_loader, EPOCHS, DEVICE, lr=1e-3)
cu.plot_history(history)
model.eval()


## 4. Select a correct prediction and a mistake when available

We inspect held-out examples for this fixed-model diagnostic lesson. Once test examples
influence a model revision, use a fresh test set for the revised model's final evaluation.
Small smoke runs might contain no correct examples, or no mistakes: the selection handles both.


In [ ]:
logits, labels = cu.predict(model, test_loader, DEVICE)
probabilities = logits.softmax(dim=1)
predictions = probabilities.argmax(dim=1)
correct_ids = torch.where(predictions == labels)[0]
wrong_ids = torch.where(predictions != labels)[0]
selected = []
if len(correct_ids):
    selected.append(int(correct_ids[0]))
if len(wrong_ids):
    selected.append(int(wrong_ids[0]))
if not selected:
    selected = [0]
print(f'Test accuracy: {(predictions == labels).float().mean():.3f}')
print('Selected dataset indices:', selected)
for idx in selected:
    print(idx, 'true:', class_names[int(labels[idx])],
          'predicted:', class_names[int(predictions[idx])],
          'confidence:', round(float(probabilities[idx].max()), 3))


## 5. Input saliency: sensitivity of one logit to the pixels

For target class $c$, calculate $|\partial z_c/\partial x|$.
This measures a local derivative, not what caused the real-world object to belong to a class.
We explain a **logit**, avoiding an additional softmax derivative.

`model.eval()` turns off training behavior such as dropout. It does **not** disable gradients.
Explanation code must not run inside `torch.inference_mode()`; this function explicitly enables gradients.


In [ ]:
def normalize_map(values):
    values = values.detach().float().cpu()
    lo, hi = values.min(), values.max()
    return (values - lo) / (hi - lo).clamp_min(1e-8)

def input_saliency(model, image, target_class):
    model.eval()
    with torch.enable_grad():
        x = image.unsqueeze(0).to(DEVICE).detach().requires_grad_(True)
        score = model(x)[0, int(target_class)]
        gradient, = torch.autograd.grad(score, x)
    # max across channels also supports RGB images
    return normalize_map(gradient[0].abs().amax(dim=0))

sample_idx = selected[0]
sample, true_class = test_ds[sample_idx]
target_class = int(predictions[sample_idx])
saliency = input_saliency(model, sample, target_class)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(sample[0], cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Input: {class_names[true_class]}')
axes[1].imshow(saliency, cmap='magma', vmin=0, vmax=1)
axes[1].set_title(f'Sensitivity: {class_names[target_class]}')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 6. Grad-CAM: explain with spatial feature maps

For activations $A^k$ in a convolutional layer, average the target-logit gradients over
height and width to obtain channel weights $\alpha_k$. Then form
$\operatorname{ReLU}(\sum_k \alpha_k A^k)$ and resize it for display.
The original feature grid is coarse; interpolation does not add localization precision.
This implements the core method from the [Grad-CAM paper](https://arxiv.org/abs/1610.02391).

A forward hook saves the activations without detaching them. `autograd.grad` obtains
their gradients without accumulating parameter gradients. `finally` removes the hook,
including when an error occurs.


In [ ]:
def grad_cam(model, image, target_class, target_layer):
    model.eval()
    captured = {}

    def capture_output(module, inputs, output):
        captured['activation'] = output

    handle = target_layer.register_forward_hook(capture_output)
    try:
        with torch.enable_grad():
            x = image.unsqueeze(0).to(DEVICE).detach().requires_grad_(True)
            score = model(x)[0, int(target_class)]
            activation = captured['activation']
            gradients, = torch.autograd.grad(score, activation)
            weights = gradients.mean(dim=(2, 3), keepdim=True)
            coarse = (weights * activation).sum(dim=1, keepdim=True).relu()
            resized = F.interpolate(coarse, size=image.shape[-2:],
                                    mode='bilinear', align_corners=False)
        return normalize_map(resized[0, 0]), tuple(activation.shape[-2:])
    finally:
        handle.remove()

convolution_layers = [m for m in model.features.modules() if isinstance(m, nn.Conv2d)]
target_layer = convolution_layers[-1]
cam, grid_size = grad_cam(model, sample, target_class, target_layer)
print('Target layer:', target_layer)
print('Original explanation grid:', grid_size, 'Displayed image:', sample.shape[-2:])


## 7. Compare explanations for correct and incorrect predictions

The map is conditioned on the requested class. A confident mistake can still produce
a convincing heatmap. Use the title, true class, and predicted class when presenting it.
A flat all-zero Grad-CAM map can occur and is displayed as such, not silently replaced.


In [ ]:
fig, axes = plt.subplots(len(selected), 3, figsize=(10, 3.3 * len(selected)), squeeze=False)
for row, idx in enumerate(selected):
    image, true_id = test_ds[idx]
    predicted_id = int(predictions[idx])
    local_saliency = input_saliency(model, image, predicted_id)
    local_cam, _ = grad_cam(model, image, predicted_id, target_layer)
    axes[row, 0].imshow(image[0], cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(f'True: {class_names[true_id]}\nPred: {class_names[predicted_id]}')
    axes[row, 1].imshow(local_saliency, cmap='magma', vmin=0, vmax=1)
    axes[row, 1].set_title('Absolute input gradient')
    axes[row, 2].imshow(image[0], cmap='gray', vmin=0, vmax=1)
    axes[row, 2].imshow(local_cam, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[row, 2].set_title('Grad-CAM for predicted class')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout()
cu.ARTIFACTS.mkdir(parents=True, exist_ok=True)
attribution_path = cu.ARTIFACTS / '09_attributions.png'
fig.savefig(attribution_path, dpi=140, bbox_inches='tight')
plt.show()
print('Saved:', attribution_path)


## 8. Change the question by changing the target class

For a mistake, compare the predicted class with the true class. Otherwise use the
second-highest scoring class. The same pixels can support different model questions.


In [ ]:
sample_idx = int(wrong_ids[0]) if len(wrong_ids) else selected[0]
sample, true_class = test_ds[sample_idx]
target_class = int(predictions[sample_idx])
alternative_class = (int(true_class) if target_class != int(true_class)
                     else int(probabilities[sample_idx].topk(2).indices[1]))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, class_id in zip(axes, [target_class, alternative_class]):
    heatmap, _ = grad_cam(model, sample, class_id, target_layer)
    ax.imshow(sample[0], cmap='gray', vmin=0, vmax=1)
    ax.imshow(heatmap, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    ax.set_title('Target: ' + class_names[class_id])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 9. Occlusion: change pixels and measure the score change

Replace a small patch with black and measure **original target logit minus occluded
target logit**. Positive values mean the intervention reduced that score. Negative values
mean it increased the score. Unlike the gradient methods, this needs only forward passes.

Occlusion is a controlled input intervention, but the square mask may itself be unusual
for the data. Conclusions depend on patch size and replacement value.


In [ ]:
@torch.no_grad()
def occlusion_map(model, image, target_class, patch_size=7, stride=7, fill=0.0):
    model.eval()
    x = image.unsqueeze(0).to(DEVICE)
    baseline = model(x)[0, int(target_class)]
    h, w = image.shape[-2:]
    changes = torch.zeros(h, w, device=DEVICE)
    counts = torch.zeros_like(changes)
    for top in range(0, h, stride):
        for left in range(0, w, stride):
            bottom, right = min(top + patch_size, h), min(left + patch_size, w)
            altered = x.clone()
            altered[:, :, top:bottom, left:right] = fill
            delta = baseline - model(altered)[0, int(target_class)]
            changes[top:bottom, left:right] += delta
            counts[top:bottom, left:right] += 1
    return (changes / counts.clamp_min(1)).cpu()

occlusion = occlusion_map(model, sample, target_class)
limit = max(float(occlusion.abs().max()), 1e-8)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(occlusion, cmap='coolwarm', vmin=-limit, vmax=limit)
ax.set_title('Target-logit drop after black-patch occlusion')
ax.axis('off')
fig.colorbar(im, ax=ax, label='Original score - occluded score')
plt.tight_layout()
plt.show()


## 10. A sanity check: compare trained and random weights

An explanation intended to describe a learned model should be examined for sensitivity
to its parameters. Keep the same image and target class, then compare the trained model
with an independently initialized model. One image is a demonstration, not a formal
validation of an explanation method.


In [ ]:
cu.seed_everything(123)
random_model = cu.SmallCNN(num_classes=len(class_names), in_channels=1).to(DEVICE).eval()
random_layer = [m for m in random_model.features.modules() if isinstance(m, nn.Conv2d)][-1]
trained_map, _ = grad_cam(model, sample, target_class, target_layer)
random_map, _ = grad_cam(random_model, sample, target_class, random_layer)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, heatmap, title in zip(axes, [trained_map, random_map], ['Trained model', 'Random model']):
    ax.imshow(heatmap, cmap='magma', vmin=0, vmax=1)
    ax.set_title(title + ': same target class')
    ax.axis('off')
plt.tight_layout()
plt.show()
del random_model


## 11. Practical exercises

1. Run occlusion with `patch_size=4, stride=4`, then with `fill=0.5`. Explain what changes.
2. Use the first convolutional layer for Grad-CAM. Compare spatial resolution and meaning.
3. Find a high-confidence mistake and inspect both the predicted and true class.
4. Record one observation the maps support and one claim they cannot establish.

<details><summary>Suggested solutions and discussion</summary>

Call `occlusion_map(model, sample, target_class, patch_size=4, stride=4, fill=0.5)`.
Smaller patches perform more forward passes and measure a different intervention.
Call `grad_cam(model, sample, target_class, convolution_layers[0])` for the earlier layer.
For exercise 3, sort `probabilities.max(1).values[wrong_ids]` in descending order;
skip it explicitly if there are no errors. A map may indicate score sensitivity to an
image region; it does not establish causal reasoning, fairness, or deployment safety.

</details>

**Demonstration checkpoint:** show an input, state the target class, compute an attribution,
change the target, and test a region by occlusion. Explain why the methods may disagree.


## References and next lesson

- [Grad-CAM original paper](https://arxiv.org/abs/1610.02391)
- [Captum attribution-method comparison](https://captum.ai/docs/algorithms_comparison_matrix)
- [PyTorch autograd documentation](https://docs.pytorch.org/docs/stable/autograd.html)
- [Sanity Checks for Saliency Maps](https://arxiv.org/abs/1810.03292)

Next: [10 · Vision transformers](10_vision_transformers.ipynb).
